## NEXTBUY - From Raw Data to Smart Decisions

---

### Contexte

Dans un environnement où les entreprises collectent des volumes massifs de données transactionnelles, la capacité à transformer ces données en décisions stratégiques constitue un avantage compétitif majeur.

Nous disposons d’un ensemble de données comprenant :

- Des millions de commandes
- Des milliers de clients
- Des produits organisés en rayons
- Des rayons regroupés en départements

Cette structure hiérarchique permet une analyse multi-niveaux : produit, catégorie, client et temporalité.

---

### Objectifs du Projet

Notre démarche repose sur deux axes principaux :

#### Analyse Exploratoire des Données (EDA)

Identifier des insights exploitables permettant :

- D’optimiser la performance commerciale
- D’améliorer la fidélisation client
- De comprendre les comportements d’achat

#### Modélisation Prédictive

Construire des modèles capables de :

- Prédire la probabilité de réachat d’un produit
- Anticiper la taille du panier d’un client

## Préparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products.csv")
products = pd.read_csv("data/products.csv")
aisles = pd.read_csv("data/aisles.csv")
departments = pd.read_csv("data/departments.csv")

df = (
    order_products
    .merge(orders, on="order_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(aisles, on="aisle_id", how="left")
    .merge(departments, on="department_id", how="left")
)

## Nettoyage des données

In [ ]:
#------------------------------------------------------   NETTOYAGE

#dropna ou filna

df = df.dropna(subset=["product_id"])
df["reordered"] = df["reordered"].fillna(0)
df = df.dropna(subset=["user_id"])

# colonnes à convertir en entier
cols_to_int = [
    "product_id", 
    "add_to_cart_order", 
    "reordered", 
    "user_id", 
    "order_number", 
    "order_dow", 
    "aisle_id", 
    "department_id"
]

# convertir en entier  (int64 pour les nan)
df[cols_to_int] = df[cols_to_int].astype("Int64")

# Remplacements colonnes valeurs: 
df["order_number"], df["order_dow"] = df["order_dow"], df["order_number"]

df["order_number"], df["order_hour_of_day"] = df["order_hour_of_day"], df["order_number"]

# export du dataframe fusionné + nettoyé
df.to_csv("data/merged_clean.csv", index=False)

#------------------------------------------------------------------------- NETTOYAGE

## Nombre total de commandes

In [ ]:
total_orders = df["order_id"].nunique()
print("Nombre total de commandes :", total_orders)

## Nombre total de produits

In [ ]:
total_products = df["product_id"].nunique()
print("Nombre total de produits :", total_products)

## Produits par commande

In [ ]:
product_par_basket = df.groupby("order_id")["product_id"].count()

## Panier moyen

In [ ]:
average_basket = product_par_basket.mean()
print("Panier moyen :", round(average_basket, 1))

## Top des produits les plus vendus

In [ ]:
lemost = df.groupby(["product_id", "product_name"]).size().sort_values(ascending=False)
print(lemost.head())

# Départements les plus populaires

In [ ]:
dept_popularity = df.groupby(["department_id", "department"]).size().reset_index(name="nb_ventes").sort_values(by="nb_ventes", ascending=False)
dept_popularity.head()

## Commandes par jours

In [ ]:
# compter le nombre de commandes par jour (0=Dimanche)
orders_per_day = (
	orders.dropna(subset=["order_dow"])
	.assign(order_dow=lambda x: x["order_dow"].astype(int))
	.loc[lambda x: x["order_dow"].between(0, 6)]
	.groupby("order_dow")["order_id"]
	.nunique()
	.reindex(range(7), fill_value=0)
)

# graphique
orders_per_day.plot(kind="bar")

plt.title("Nombre de commandes par jour de la semaine")
plt.xlabel("Jour de la semaine (0 = Dimanche)")
plt.ylabel("Nombre de commandes")

plt.show()

## Reorder selon la position dans le panier

In [ ]:
# calcul du reorder
reorder_by_position = (
    order_products.groupby("add_to_cart_order")["reordered"]
    .mean()
    .reset_index()
)

# 15 remirs
reorder_by_position = reorder_by_position.head(15)

# en pourcentage
reorder_by_position["reordered_pct"] = reorder_by_position["reordered"] * 100

# graphique
plt.figure(figsize=(8,5))
plt.plot(reorder_by_position["add_to_cart_order"],
         reorder_by_position["reordered_pct"],
         marker="o", linestyle="-", color="blue")

plt.xlabel("Position dans le panier")
plt.ylabel("Taux de reorder (%)")
plt.title("Taux de reorder selon la position dans le panier")
plt.grid(True)
plt.show()

##### l'analyse du taux de reorder par position dans le panier montre que les produits ajoutés en début de panier sont reorder plus fréquemment que ceux placés en fin de panier. Cela indique que les clients achètent régulièrement leurs articles favoris en premier, ce qui reflète à la fois leurs habitudes et la priorité accordée à certains produits dans leurs commandes.

# Les produits dans les petits paniers sont ils les plus reorder ?

In [ ]:
# taille du panier
cart_size = order_products.groupby("order_id").size().reset_index(name="cart_size")

# taux reorder moyen par commande
reorder_rate = order_products.groupby("order_id")["reordered"].mean().reset_index()

# fusionner
basket_df = cart_size.merge(reorder_rate, on="order_id")

# taux moyen de reorder selon la taille du panier
result = basket_df.groupby("cart_size")["reordered"].mean().reset_index()

# 20 premiers
result = result.head(20)

result["reordered"] = result["reordered"] * 100
# graphique
plt.figure()
plt.plot(result["cart_size"], result["reordered"], marker="o")

plt.xlabel("Taille du panier")
plt.ylabel("Taux de reorder")
plt.title("Relation entre taille du panier et fidélité")
plt.show()

##### Oui, nous observons que les plus petits paniers, qui contiennent souvent des articles essentiels comme l'eau, les œufs, le pain ou le lait, présentent un taux de reorder supérieur à 60 %. Cela signifie que ces produits sont rachetés plus de la moitié du temps, ce qui reflète leur caractère indispensable dans les habitudes d'achat des clients.

## Premier produit mit dans le panier

In [ ]:

first_items = df[df["add_to_cart_order"] == 1]
top_first = first_items["product_name"].value_counts().head(10)

# graphique camembert
plt.figure(figsize=(8, 8))
plt.pie(top_first.values, labels=top_first.index, autopct="%1.1f%%", startangle=90)
plt.title("Top 10 first items put in cart")
plt.show()

## Optimiser le placement produit en magasin

 - Les rayons à la fois très vendus et très reachetés méritent les meilleures places (entrée, allées centrales, hauteur des yeux).

In [ ]:
# score de placement par rayon (aisle)

# df (aisle + reordered)
aisle_stats = (
    df.groupby("aisle")
    .agg(volume=("product_id", "count"), reorder_rate=("reordered", "mean"))
    .reset_index()
)

# normaliser (sinon ca fausse tout)
aisle_stats["vol_norm"] = (aisle_stats["volume"] - aisle_stats["volume"].min()) / (aisle_stats["volume"].max() - aisle_stats["volume"].min())
aisle_stats["ror_norm"] = (aisle_stats["reorder_rate"] - aisle_stats["reorder_rate"].min()) / (aisle_stats["reorder_rate"].max() - aisle_stats["reorder_rate"].min())

# score de l'allée = volume × reorder
aisle_stats["placement_score"] = aisle_stats["vol_norm"] * aisle_stats["ror_norm"]

# top 10 rayons 
top_aisles = aisle_stats.sort_values("placement_score", ascending=False).head(10)

# graphique
plt.figure(figsize=(9, 5))
plt.barh(top_aisles["aisle"][::-1], top_aisles["placement_score"][::-1])
plt.xlabel("Score de placement (volume × reorder normalisés)")
plt.title("Top 10 rayons à placer en priorité en magasin")
plt.tight_layout()
plt.show()

 ##### Pour maximiser les profits du magasin il faudrait placer les fruits et légumes frais à l'entrée du magasin ou dans les allées centrales, car ils attirrent beaucoup de mondes et sont souvent recommandés. Les produits comme le yaourt ou le lait, pourraient êtres placés à hauteur des yeux dans les rayons frais, pour optimiser leur visibilité et favoriser l'achat d'impulsion. Pour finir, les produits de consommation quotidienne, comme  l'eau ou le pain, doivent être placé de façon a être facilement accessibles car ils attirent beaucoup de monde aussi mais ne doivent pas bloquer le trafic.

NEXTBUY - From Raw Data to Smart Decisions
Contexte
Dans un environnement où les entreprises collectent des volumes massifs de données transactionnelles, la capacité à transformer ces données en décisions stratégiques constitue un avantage compétitif majeur.

Nous disposons d'un ensemble de données comprenant :

Des millions de commandes
Des milliers de clients
Des produits organisés en rayons
Des rayons regroupés en départements
Cette structure hiérarchique permet une analyse multi-niveaux : produit, catégorie, client et temporalité.

Objectifs du Projet
Notre démarche repose sur deux axes principaux :

Analyse Exploratoire des Données (EDA)
Identifier des insights exploitables permettant :

D'optimiser la performance commerciale
D'améliorer la fidélisation client
De comprendre les comportements d'achat
Modélisation Prédictive
Construire des modèles capables de :

Prédire la probabilité de réachat d'un produit
Anticiper la taille du panier d'un client
Business Questions & Analytical Objectives
Afin de structurer notre analyse, nous avons défini les huit questions stratégiques suivantes :

Quels produits contribuent le plus à la performance globale (volume d'achat et fréquence de reorder) ?

In [ ]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products.csv")
products = pd.read_csv("data/products.csv")

df = orders.merge(order_products, on="order_id")
df = df.merge(products, on="product_id")

volume = df.groupby("product_name").size().reset_index(name="total_purchases")
reorder_rate = df.groupby("product_name")["reordered"].mean().reset_index(name="reorder_rate")

performance = volume.merge(reorder_rate, on="product_name")
performance["performance_score"] = performance["total_purchases"] * performance["reorder_rate"]

top_products = performance.sort_values("performance_score", ascending=False).head(10)

top_products

Interprétation des résultats
Le tableau final affiche les 10 produits ayant le score de performance le plus élevé.

Ce score combine :

le nombre total d'achats (total_purchases)
la proportion moyenne de reorder (reorder_rate)
Un score élevé signifie qu'un produit est à la fois :

très acheté
fréquemment racheté
Justification méthodologique (par rapport au code)
Les fichiers sont fusionnés afin d'associer chaque achat à son produit et à son indicateur de reorder.
Le volume est calculé avec un groupby().size() pour compter toutes les occurrences d'achat.
Le taux de reorder est obtenu avec mean() sur la variable binaire reordered (0/1), ce qui donne directement une proportion.
Le score final est construit comme un produit multiplicatif (volume × reorder_rate) pour pondérer la fidélité par la popularité.
Le tri décroissant permet d'identifier les produits ayant l'impact global le plus fort.
Cette méthode permet d'éviter de privilégier uniquement les produits très populaires ou uniquement les produits très fidèles : elle combine les deux dimensions dans un indicateur unique.

Quels produits présentent le taux de reorder le plus élevé et quels facteurs peuvent l'expliquer ?

In [ ]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products.csv")
products = pd.read_csv("data/products.csv")

df = orders.merge(order_products, on="order_id")
df = df.merge(products, on="product_id")

product_stats = df.groupby("product_name").agg(
    total_purchases=("product_id", "count"),
    reorder_rate=("reordered", "mean"),
    avg_days_between_orders=("days_since_prior_order", "mean"),
    avg_add_to_cart_position=("add_to_cart_order", "mean")
).reset_index()

product_stats = product_stats[product_stats["total_purchases"] > 100]

top_reorder = product_stats.sort_values("reorder_rate", ascending=False).head(10)

top_reorder

Interprétation des résultats
Les produits affichés ont le taux moyen de reorder le plus élevé parmi ceux ayant un volume suffisant (>100 achats pour éviter le bruit statistique).

Un taux proche de 1 signifie que le produit est presque systématiquement racheté après un premier achat.

Les colonnes supplémentaires permettent d'explorer des facteurs explicatifs :

avg_days_between_orders : indique si le produit est acheté de manière régulière.
avg_add_to_cart_position : une position faible suggère un produit prioritaire ou essentiel.
total_purchases : permet de distinguer un produit niche très fidèle d'un produit massif et fidèle.
Justification méthodologique
Le groupby permet d'agréger les métriques par produit.
La moyenne de reordered donne directement le taux de reorder car la variable est binaire (0/1).
Un filtre sur total_purchases est appliqué pour éviter qu'un produit avec très peu d'achats mais 100 % de reorder apparaisse artificiellement en tête.
L'ajout de variables comportementales (temps entre commandes, position dans le panier) permet d'explorer des corrélations possibles sans encore construire de modèle explicatif.
Cette approche identifie les produits les plus fidèles et fournit des indicateurs permettant d'analyser les mécanismes associés à cette fidélité.

À quelles heures les clients passent-ils le plus de commandes, et quels produits dominent ces créneaux ?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/merged_clean.csv")

# Garder uniquement les heures valides (0 à 23)
df = df[df["order_hour_of_day"].between(0, 23)].copy()

# Nombre de commandes par heure
orders_by_hour = (
    df.groupby("order_hour_of_day")["order_id"]
    .nunique()
    .reset_index(name="total_orders")
    .sort_values("order_hour_of_day")
)

# Heure avec le plus de commandes
peak_hour = int(orders_by_hour.loc[orders_by_hour["total_orders"].idxmax(), "order_hour_of_day"])

# Produits dominants pendant l'heure de pic
top_products_hour = (
    df[df["order_hour_of_day"] == peak_hour]
    .groupby("product_name")
    .size()
    .reset_index(name="purchases")
    .sort_values("purchases", ascending=False)
    .head(6)
)

# Graphique 1 : commandes par heure
plt.figure(figsize=(10, 4))
plt.bar(orders_by_hour["order_hour_of_day"], orders_by_hour["total_orders"], color="skyblue")
plt.axvline(peak_hour, color="red", linestyle="--", label=f"Heure de pic : {peak_hour}h")
plt.title("Nombre de commandes par heure")
plt.xlabel("Heure")
plt.ylabel("Nombre de commandes")
plt.xticks(range(0, 24))
plt.legend()
plt.show()

# Graphique 2 : camembert des produits dominants à l'heure de pic
plt.figure(figsize=(8, 8))
plt.pie(
    top_products_hour["purchases"],
    labels=top_products_hour["product_name"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title(f"Produits dominants à {peak_hour}h")
plt.show()

print(f"Heure avec le plus de commandes : {peak_hour}h")
top_products_hour

Interprétation des résultats
peak_day indique le jour de la semaine générant le plus grand nombre de commandes uniques.
peak_hour identifie l'heure avec la plus forte activité.
Les tableaux top_products_day et top_products_hour montrent les produits les plus achetés durant ces créneaux.

On observe généralement :

Un pic en fin de semaine.
Une concentration des commandes en fin de matinée ou début de soirée.
Une domination de produits frais ou essentiels pendant ces périodes de forte activité.
Justification méthodologique
Le nombre de commandes est mesuré avec nunique() sur order_id pour éviter de compter plusieurs fois une même commande contenant plusieurs produits.
Le tri décroissant permet d'identifier le jour et l'heure à volume maximal.
Une fois les créneaux identifiés, un filtrage conditionnel isole les transactions correspondantes.
Un groupby().size() permet d'identifier les produits dominants sur ces périodes spécifiques.
Cette méthode permet d'analyser simultanément la dimension temporelle (quand les clients commandent) et la dimension produit (quoi ils achètent à ces moments).

Quels produits sont fréquemment achetés ensemble ?

In [ ]:
import pandas as pd
from itertools import combinations

df = pd.read_csv("data/merged_clean.csv")


basket = df.groupby("order_id")["product_name"].apply(list)

pair_counts = {}

for products_list in basket:
    unique_products = set(products_list)
    for pair in combinations(sorted(unique_products), 2):
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

pairs_df = pd.DataFrame(
    [(p[0], p[1], c) for p, c in pair_counts.items()],
    columns=["product_1", "product_2", "co_purchase_count"]
)

top_pairs = pairs_df.sort_values("co_purchase_count", ascending=False).head(10)

top_pairs

Interprétation des résultats
Le tableau affiche les paires de produits apparaissant le plus souvent dans une même commande.

Un co_purchase_count élevé signifie que ces deux produits sont régulièrement présents ensemble dans le panier.

Ces associations reflètent généralement :

Des produits complémentaires (ex. chips + soda).
Des produits d'une même catégorie.
Des habitudes d'achat récurrentes.
Justification méthodologique
Les produits sont regroupés par order_id pour reconstruire chaque panier.
Les combinaisons de taille 2 sont générées pour chaque panier afin d'identifier toutes les paires possibles.
L'utilisation d'un set évite de compter deux fois un même produit dans une commande.
Un comptage global permet de mesurer la fréquence d'apparition de chaque paire.
Le tri décroissant identifie les associations les plus fréquentes.
Cette méthode repose sur une logique de co-occurrence simple permettant d'identifier des relations produit-produit sans modèle probabiliste.

Quels profils clients peuvent être identifiés à partir de leurs comportements d'achat ?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv("data/merged_clean.csv")

user_features = df.groupby("user_id").agg(
    total_orders=("order_number", "max"),
    avg_days_between_orders=("days_since_prior_order", "mean"),
    avg_cart_position=("add_to_cart_order", "mean"),
    reorder_rate=("reordered", "mean"),
    avg_hour=("order_hour_of_day", "mean")
).reset_index()

user_features["avg_days_between_orders"] = user_features["avg_days_between_orders"].fillna(0)
user_features = user_features.dropna()

X = user_features.drop("user_id", axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
user_features["cluster"] = kmeans.fit_predict(X_scaled)

cluster_profiles = user_features.groupby("cluster").mean(numeric_only=True)
cluster_sizes = user_features["cluster"].value_counts().sort_index()
cluster_reorder = user_features.groupby("cluster")["reorder_rate"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(cluster_sizes.index.astype(str), cluster_sizes.values, color="skyblue")
axes[0].set_title("Nombre de clients par cluster")
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Nombre de clients")

axes[1].bar(cluster_reorder.index.astype(str), cluster_reorder.values, color="salmon")
axes[1].set_title("Taux moyen de reorder par cluster")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Reorder moyen")

plt.tight_layout()
plt.show()

cluster_profiles.round(2)

Interprétation des résultats

Chaque cluster représente un profil d'achat différent.

- **Cluster 0 — acheteurs réguliers et assez fidèles**  
  Ces clients ont un nombre de commandes correct, un taux de reorder élevé (environ 0.53) et commandent plutôt en début d'après-midi. Ils semblent avoir des habitudes d'achat déjà bien installées.

- **Cluster 1 — acheteurs du soir, moins fidèles**  
  Ce groupe commande surtout plus tard dans la journée, avec un taux de reorder plus faible (environ 0.27). Cela peut correspondre à des clients plus occasionnels ou qui varient davantage leurs achats.

- **Cluster 2 — meilleurs clients / clients très fidèles**  
  C'est le groupe le plus intéressant : il est le plus nombreux et possède le taux de reorder le plus élevé (environ 0.62). Ces clients achètent plus souvent les mêmes produits et semblent avoir une routine d'achat stable, plutôt le matin.

- **Cluster 3 — petits acheteurs occasionnels**  
  Ce cluster a le plus faible nombre total de commandes (environ 1.68). Les clients de ce groupe paraissent plus récents ou moins engagés, avec une fidélité moyenne.

Lecture business simple

- Le **cluster 2** correspond aux clients les plus précieux à fidéliser.
- Le **cluster 0** regroupe aussi de bons clients, mais un peu moins engagés.
- Le **cluster 1** pourrait être ciblé avec des recommandations personnalisées pour augmenter le reorder.
- Le **cluster 3** peut être travaillé avec des offres de réactivation ou de bienvenue.

Les profils sont construits à partir de plusieurs indicateurs comportementaux :
- nombre total de commandes,
- délai moyen entre commandes,
- position moyenne dans le panier,
- taux de reorder,
- heure moyenne de commande.

Le clustering permet ensuite de regrouper automatiquement les clients qui se ressemblent afin de faire émerger des segments interprétables d'un point de vue marketing.

Peut-on prédire la taille du panier d'un client en fonction de son historique ?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("data/merged_clean.csv")

# 1 ligne = 1 commande
cart_size = df.groupby("order_id").size().reset_index(name="cart_size")
order_level = df.groupby("order_id").agg(
    total_orders=("order_number", "max"),
    avg_days_between_orders=("days_since_prior_order", "mean"),
    reorder_rate=("reordered", "mean"),
    avg_hour=("order_hour_of_day", "mean")
).reset_index()

order_level = order_level.merge(cart_size, on="order_id")

features = ["total_orders", "avg_days_between_orders", "reorder_rate", "avg_hour"]
order_level[features] = order_level[features].fillna(0)

X = order_level[features]
y = order_level["cart_size"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
lin_pred = lin_model.predict(X_test)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

lin_mae = mean_absolute_error(y_test, lin_pred)
lin_r2 = r2_score(y_test, lin_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

metrics_df = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [lin_mae, rf_mae],
    "R2": [lin_r2, rf_r2]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(metrics_df["Model"], metrics_df["MAE"], color=["#4C78A8", "#F58518"])
axes[0].set_title("MAE (plus bas = meilleur)")
axes[0].set_ylabel("MAE")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(metrics_df["Model"], metrics_df["R2"], color=["#4C78A8", "#F58518"])
axes[1].set_title("R² (plus haut = meilleur)")
axes[1].set_ylabel("R²")
axes[1].tick_params(axis="x", rotation=15)

plt.suptitle("Comparaison des modèles - Prédiction de la taille du panier")
plt.tight_layout()
plt.show()

metrics_df

Interprétation des résultats
Les métriques retournées sont :

MAE Linear Regression
R² Linear Regression
MAE Random Forest
R² Random Forest
Le MAE mesure l'erreur moyenne en nombre d'articles. Le R² indique la proportion de variance expliquée.

Si le Random Forest présente un MAE plus faible et un R² plus élevé, cela signifie que la taille du panier dépend de relations non linéaires entre les variables comportementales.

Un R² modéré (0.3–0.5) indique que l'historique explique partiellement le panier. Un R² faible suggère que d'autres variables non présentes influencent fortement la taille du panier.

Justification méthodologique
La taille du panier est calculée par un comptage du nombre de produits par order_id.

Des variables historiques sont agrégées au niveau utilisateur pour représenter le comportement global.

La variable cible étant continue, une régression supervisée est adaptée.

Un split train/test permet d'évaluer la capacité de généralisation.

Deux modèles sont comparés :

Régression linéaire pour tester une relation simple.
Random Forest pour capturer des interactions complexes.
Cette approche permet d'évaluer si le comportement passé du client contient un signal prédictif sur la taille future de ses paniers.

Dataset preparation for Machine Learning
Présentation des datasets
Le projet utilise plusieurs fichiers décrivant les commandes, les produits et les catégories. orders contient les informations sur les commandes, order_products relie les produits aux commandes et indique si un produit est reorder, et products, aisles et departments décrivent les produits et leurs catégories.

Ces tables sont reliées par order_id, product_id et user_id afin de reconstituer l'historique des achats.

In [ ]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products.csv")
products = pd.read_csv("data/products.csv")
aisles = pd.read_csv("data/aisles.csv")
departments = pd.read_csv("data/departments.csv")

df = orders.merge(order_products, on="order_id")
df = df.merge(products, on="product_id")
df = df.merge(aisles, on="aisle_id")
df = df.merge(departments, on="department_id")

df.to_csv("data/merged_clean.csv", index=False)

print("Shape :", df.shape)
df.head()

Model 1 : Reorder prediction
Le premier modèle vise à prédire si un produit sera racheté lors d'une prochaine commande (reordered). Chaque observation correspond à un produit présent dans une commande. Les variables utilisées décrivent le contexte d'achat, comme l'heure de la commande, le jour de la semaine, le délai depuis la commande précédente et la position du produit dans le panier. Deux modèles sont entraînés et évalués : une régression logistique et un Random Forest Classifier.

In [ ]:
import pandas as pd

df = pd.read_csv("data/merged_clean.csv")

features_model1 = [
    "user_id",
    "product_id",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "add_to_cart_order",
    "aisle_id",
    "department_id"
]

target_model1 = "reordered"

model_ready_reorder = df[features_model1 + [target_model1]].dropna()

print("Dataset Model 1 shape :", model_ready_reorder.shape)

Interprétation des résultats
Chaque observation correspond à un produit présent dans une commande. La variable cible est reordered, qui indique si le produit a déjà été acheté auparavant par le client.

Model 2 : Cart size prediction
Le second modèle vise à prédire la taille du panier d'une commande (cart_size). Dans ce cas, chaque observation correspond à une commande (order_id). Les variables utilisées décrivent les caractéristiques de la commande, notamment le jour, l'heure et le délai depuis la commande précédente. Deux modèles sont utilisés : une régression linéaire et un Random Forest Regressor.

In [ ]:
cart_size = df.groupby("order_id").size().reset_index(name="cart_size")

orders_features = df.groupby("order_id").agg({
    "user_id": "first",
    "order_dow": "first",
    "order_hour_of_day": "first",
    "days_since_prior_order": "first"
}).reset_index()

model_ready_cart = orders_features.merge(cart_size, on="order_id").dropna()

print("Dataset Model 2 shape :", model_ready_cart.shape)

Interpétation des résultats
Chaque observation correspond à une commande (order_id). La variable cible cart_size représente le nombre total de produits dans le panier.

Justification
Le dataset fusionné est transformé en deux datasets adaptés aux deux tâches de machine learning. Pour la classification, l'unité d'observation est le produit dans une commande. Pour la régression, les données sont agrégées au niveau de la commande afin de calculer la taille du panier.

Model comparison
Les performances des modèles sont comparées afin d'identifier celui qui explique le mieux les comportements d'achat. Pour la classification, les métriques utilisées sont l'accuracy et le ROC-AUC. Pour la régression, les modèles sont évalués à l'aide du MAE et du R². Cette comparaison permet de déterminer si des modèles plus complexes apportent une amélioration par rapport aux modèles plus simples.

--

Deux types de modèles ont été utilisés dans ce projet afin de répondre aux deux tâches de machine learning. Pour la prédiction du reorder, une régression logistique et un Random Forest ont été entraînés. La régression logistique est un modèle simple qui permet d'établir une relation linéaire entre les variables et la probabilité de rachat d'un produit. Le Random Forest, en revanche, est capable de capturer des relations plus complexes entre les variables grâce à l'utilisation de plusieurs arbres de décision.

Pour la prédiction de la taille du panier, une régression linéaire et un Random Forest Regressor ont été utilisés. La régression linéaire fournit une estimation simple basée sur une relation linéaire entre les variables explicatives et la taille du panier. Le Random Forest Regressor permet quant à lui de modéliser des comportements d'achat plus complexes.

La comparaison des performances permet d'identifier le modèle le plus adapté pour chaque tâche. En général, les modèles Random Forest offrent de meilleures performances car ils peuvent capturer des relations non linéaires entre les variables.

Conclusion
Ce projet a permis d'analyser les comportements d'achat des clients à partir du dataset NextBuy. L'analyse exploratoire a permis d'identifier les produits les plus populaires, les moments où les clients passent le plus de commandes et certaines relations entre les produits achetés ensemble.

Les modèles de machine learning montrent qu'il est possible de prédire certains comportements d'achat, comme la probabilité qu'un produit soit racheté ou la taille du panier d'une commande. Ces résultats permettent de mieux comprendre les habitudes des clients et peuvent être utilisés pour améliorer les recommandations produits ou optimiser les stratégies commerciales.